# 05 Gamma forecasting

Asks what the correction is worth downstream: seven-day-ahead net-load forecasts for one station trained on raw, M9-corrected and manually corrected history, compared on the September 2024 test month.

Abbreviations used here: **RPF** is reverse power flow, the condition where a distribution substation exports power because rooftop solar exceeds local demand; a *wrong RPF sign* is a meter recording that stores the export as an import. **M7** is the deterministic threshold rule, **M8** the two-stage XGBoost classifier and **M9** the counterfactual bridge method of the `pynrpf` package. **MW** and **MWh** are megawatts and megawatt-hours; one interval is 15 minutes, and a *slot* counts intervals from midnight (slot 24 is 06:00).

**Inputs.** `dataset/final/dataset_gamma.parquet` and the M9 decisions in `results/04_metrics/site_days.parquet`.

**Outputs.** `results/05_gamma/`: the three series, the data-error and forecast-error tables, the forecast impact table, the fit audit and four figures; `results/manifests/05_gamma.json`.

**Runtime.** About a minute.

**Steps.**

1. Setup.
2. Apply M9's held-out decisions to the Gamma station and forecast under three data conditions.
3. Read the impact table and the example week.

## 1. Setup

Locate the article folder, import the paper code and load the configuration. Loading the configuration verifies the SHA-256 of every dataset, so a wrong or edited data file stops the run here. `CONFIG` is the one knob: point it at another YAML to run a variant into another folder.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display


def article_root() -> Path:
    """publication/2_journal_article, found from this folder, JupyterLab's root or the repository root."""
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "paper" / "stages.py").exists():
            return candidate
        nested = candidate / "publication" / "2_journal_article"
        if (nested / "paper" / "stages.py").exists():
            return nested
    raise FileNotFoundError("Could not locate publication/2_journal_article.")


ARTICLE = article_root()
sys.path.insert(0, str(ARTICLE))
from paper import config, results, stages  # noqa: E402

CONFIG = ARTICLE / "config" / "evaluation.yaml"   # point this at another configuration to run a variant
SETTINGS = config.load(CONFIG)                     # verifies the dataset hashes before anything runs
RESULTS = SETTINGS.output_root()
print("results folder:", RESULTS.relative_to(ARTICLE))

## 2. Forecast under three data conditions

The Gamma station's readings are corrected with M9's held-out decisions (no label enters), and with the manual reference. Three forecasters (seasonal naive, linear regression, XGBoost) are fitted on each series with the settings of the configuration and scored on the test month.

In [ ]:
out = stages.gamma(SETTINGS)

## 3. The impact

Root-mean-square error in MW of the seven-day-ahead forecast on raw, M9-corrected and manually corrected history, and the week of the test month with the largest manual correction.

In [ ]:
display(out['impact'].round(3))
Image(filename=str(RESULTS / '05_gamma' / 'fig01_gamma_raw_m9_manual_example_week.png'))

## Result

The forecasting case study is on disk. All five stages of the reference run are complete.